# Cross-Sectional Mean Reversion on QQQ

---

## What This Strategy Is

We trade *relative* over/under-performance across a universe (e.g., QQQ constituents).

If a stock **underperformed** its peers today, we **buy** it.

This is **NOT** time-series mean reversion  
(price reverting to its own historical mean).

This **IS** cross-sectional mean reversion  
(relative returns reverting across assets).

---

## Book Formula (Equation 4.1): Daily Weights

The original long/short formulation is:

$$
w_{t,i} =
- \frac{
r_{t,i} - \langle r_t \rangle
}{
\sum_{k} \left| r_{t,k} - \langle r_t \rangle \right|
}
$$

### Where

- $r_{t,i}$ = daily return of stock $i$ on day $t$  
- $\langle r_t \rangle$ = cross-sectional average return  
- $\sum_k$ = sum over all stocks  

---

## Why the Minus Sign?

If:

$$
r_{t,i} > \langle r_t \rangle
$$

Then:

$$
w_{t,i} < 0
$$

→ In the original version, this shorts outperformers.

If:

$$
r_{t,i} < \langle r_t \rangle
$$

Then:

$$
w_{t,i} > 0
$$

→ This goes long underperformers.

---

## Define Cross-Sectional Deviation

Let:

$$
d_{t,i} = r_{t,i} - \langle r_t \rangle
$$

Then weights simplify to:

$$
w_{t,i} =
- \frac{d_{t,i}}
{\sum_{k} |d_{t,k}|}
$$

---

## Why the Denominator?

It normalizes exposure:

$$
\sum_i |w_{t,i}| = 1
$$

This ensures:

- Constant daily risk  
- Stable leverage  
- No volatility blow-up  

---

## Long-Only Version

Instead of allowing negative weights, we keep only positive ones:

$$
w_{t,i}^{\text{long}} =
\max \left(
0,\;
- \frac{
r_{t,i} - \langle r_t \rangle
}{
\sum_{k} \left| r_{t,k} - \langle r_t \rangle \right|
}
\right)
$$

Then re-normalize across selected stocks:

$$
w_{t,i}^{*} =
\frac{
w_{t,i}^{\text{long}}
}{
\sum_{j} w_{t,j}^{\text{long}}
}
$$

This ensures full capital deployment across only the relative losers.

---

## Portfolio Return

$$
R_{t+1} =
\sum_i w_{t,i}^{*} \, r_{t+1,i}
$$

---

## Economic Intuition

Stocks that underperform peers often mean-revert over short horizons due to:

- Liquidity imbalances  
- Temporary overreaction  
- Institutional flow distortions  
- Microstructure effects  

This strategy systematically captures short-term cross-sectional reversal.

In [5]:
import numpy as np
import pandas as pd
import yfinance as yf
import requests
import warnings
import torch

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
ANNUALIZATION = 252

START_DATE = "2024-01-01"
CHUNK = 25
MIN_OBS_FRAC = 0.90

# Hysteresis thresholds
ENTRY_Z = 1.3     # enter when z <= -ENTRY_Z (only losers)
EXIT_Z  = 0.5     # eligible to exit when z > -EXIT_Z, but only after min hold

# Minimum holding period (in trading days)
MIN_HOLD_DAYS = 2

# Transaction costs: bps per unit turnover
TC_BPS = 5
TC = TC_BPS / 10000.0

EPS = 1e-6

tickers = [
    "NVDA","AAPL","MSFT","AMZN","TSLA","META","GOOGL","WMT","GOOG","AVGO",
    "MU","COST","NFLX","PLTR","AMD","CSCO","AMAT","LRCX","TMUS","LIN",
    "PEP","INTC","AMGN","KLAC","TXN","GILD","ISRG","ADI","HON","QCOM",
    "SHOP","BKNG","APP","ASML","VRTX","PANW","CEG","INTU","CMCSA","SBUX",
    "ADBE","CRWD","WDC","MELI","MAR","ADP","CDNS","REGN","CTAS","STX",
    "CSX","ORLY","MNST","SNPS","MDLZ","AEP","DASH","WBD","MRVL","PDD",
    "ROST","PCAR","BKR","FTNT","ABNB","NXPI","MPWR","FER",   # Ferrovial: US ADR is "FER"
    "FAST","ADSK","FANG","IDXX","EA","EXC","XEL","CCEP","AXON","TRI",
    "ODFL","ALNY","PYPL","KDP","MCHP","TTWO","ROP","CPRT","MSTR",  # Strategy Inc -> MicroStrategy ticker MSTR
    "GEHC","DDOG","PAYX","INSM","CTSH","CHTR","VRSK","KHC","DXCM","WDAY",
    "ZS","CSGP","ARM","CASH",  # CASH will fail on yfinance; it will be filtered out
    "TEAM"
]

def download_adj_close(tickers, start, chunk=25):
    """
    Downloads Adj Close for tickers in chunks to avoid yfinance throttling.
    Returns a DataFrame indexed by date, columns=tickers.
    """
    out = []
    for i in range(0, len(tickers), chunk):
        batch = tickers[i:i+chunk]
        df = yf.download(
            tickers=batch,
            start=start,
            auto_adjust=False,   # we explicitly take Adj Close
            progress=False,
            group_by="column",
            threads=True,
        )

        # yfinance returns:
        # - single ticker: columns like ['Open','High',...]
        # - multiple tickers: multiindex columns (field, ticker) OR (ticker, field) depending on settings
        if df.empty:
            continue

        if isinstance(df.columns, pd.MultiIndex):
            # try both common layouts
            if ("Adj Close" in df.columns.get_level_values(0)):
                # layout: (field, ticker)
                adj = df["Adj Close"].copy()
            elif ("Adj Close" in df.columns.get_level_values(1)):
                # layout: (ticker, field)
                adj = df.xs("Adj Close", axis=1, level=1).copy()
            else:
                raise ValueError("Couldn't find Adj Close in multiindex columns.")
        else:
            # single ticker case
            if "Adj Close" in df.columns:
                adj = df[["Adj Close"]].rename(columns={"Adj Close": batch[0]})
            else:
                raise ValueError("Couldn't find Adj Close in columns.")

        out.append(adj)

    if not out:
        return pd.DataFrame()

    px = pd.concat(out, axis=1)
    px = px.sort_index()
    px = px.loc[:, ~px.columns.duplicated()]
    return px

# =========================
# Prices -> Returns
# =========================
px = download_adj_close(tickers, start=START_DATE, chunk=CHUNK)

# Stable universe + ffill
min_obs = int(MIN_OBS_FRAC * len(px))
keep = px.count()
keep = keep[keep >= min_obs].index
px = px[keep].ffill()

ret = px.pct_change()  # keep NaNs
dates = ret.index

# Torch arrays + mask (skipna-like behavior)
ret_np = ret.to_numpy(dtype=np.float32)
mask_np = ~np.isnan(ret_np)
R_np = np.nan_to_num(ret_np, nan=0.0)

R = torch.tensor(R_np, dtype=torch.float32, device=device)     # [T, N]
M = torch.tensor(mask_np.astype(np.float32), device=device)    # [T, N]
T, N = R.shape

print(f"Loaded universe: N={N}, T={T}, period {dates.min().date()} -> {dates.max().date()}")

# =========================
# Cross-sectional z-scores per day (skipna)
# =========================
count = M.sum(dim=1, keepdim=True).clamp_min(1.0)
cs_mean = (R * M).sum(dim=1, keepdim=True) / count

dev = (R - cs_mean) * M
cs_var = (dev * dev).sum(dim=1, keepdim=True) / count
cs_std = torch.sqrt(cs_var).clamp_min(EPS)

Z = (dev / cs_std) * M  # [T, N]

# =========================
# Long-only hysteresis + min hold
# We only buy losers: z <= -ENTRY_Z
# We keep holding until z > -EXIT_Z (and min hold satisfied)
#
# signal for active longs: +(-z) = -z (because z is negative -> positive signal)
# =========================
W = torch.zeros((T, N), dtype=torch.float32, device=device)

active = torch.zeros((N,), dtype=torch.bool, device=device)
hold_days = torch.zeros((N,), dtype=torch.int16, device=device)

for t in range(T):
    z_t = Z[t]                      # [N]
    valid_t = (M[t] > 0)            # [N] bool

    # increment hold counters for positions already active
    hold_days = torch.where(active, hold_days + 1, torch.zeros_like(hold_days))

    # Entry: only losers, only if inactive
    enter = (~active) & valid_t & (z_t <= -ENTRY_Z)

    # start hold at 1 on entry
    hold_days = torch.where(enter, torch.ones_like(hold_days), hold_days)
    active = active | enter

    # Exit eligibility after min hold
    can_exit = active & (hold_days >= MIN_HOLD_DAYS)

    # Exit condition for long-only losers:
    # if z has recovered above -EXIT_Z (i.e., not a big loser anymore), OR invalid
    exit_condition = (~valid_t) | (z_t > -EXIT_Z)
    exit_ = can_exit & exit_condition

    active = active & (~exit_)
    hold_days = torch.where(active, hold_days, torch.zeros_like(hold_days))

    # Build long-only signal
    signal = torch.zeros_like(z_t)
    # for active losers: signal = -z (z negative => positive)
    signal[active] = -z_t[active]

    # Normalize to total long exposure = 1
    den = torch.sum(torch.abs(signal))  # same as sum(signal) since nonnegative
    if den > 0:
        W[t] = signal / den
    else:
        W[t] = 0.0

# =========================
# No lookahead: weights decided at t apply to returns at t+1
# =========================
W_lag = torch.zeros_like(W)
W_lag[1:] = W[:-1]

gross_ret = (W_lag * R).sum(dim=1)

# =========================
# Transaction costs: turnover on HELD book
# =========================
turnover = torch.abs(W_lag - torch.roll(W_lag, shifts=1, dims=0)).sum(dim=1)
turnover[0] = 0.0

tc_cost = TC * turnover
net_ret = gross_ret - tc_cost

# =========================
# Metrics
# =========================
def perf_stats(r_1d: np.ndarray):
    r = np.nan_to_num(r_1d, nan=0.0)
    r = r[np.isfinite(r)]
    if len(r) == 0:
        return dict(apr=np.nan, sharpe=np.nan, max_dd=np.nan)

    mu = r.mean()
    sd = r.std(ddof=1)
    sharpe = (mu / sd) * np.sqrt(ANNUALIZATION) if sd > 0 else np.nan
    apr = mu * ANNUALIZATION

    equity = np.cumprod(1.0 + r)
    running_max = np.maximum.accumulate(equity)
    max_dd = np.min(equity / running_max - 1.0)

    return dict(apr=apr, sharpe=sharpe, max_dd=max_dd)

gross_np = gross_ret.detach().cpu().numpy()
net_np   = net_ret.detach().cpu().numpy()
turn_np  = turnover.detach().cpu().numpy()

stats_gross = perf_stats(gross_np)
stats_net   = perf_stats(net_np)

W_cpu = W_lag.detach().cpu().numpy()
names_held = (np.abs(W_cpu) > 0).sum(axis=1)
avg_names = names_held.mean()

eq_gross = np.cumprod(1.0 + np.nan_to_num(gross_np, nan=0.0))
eq_net   = np.cumprod(1.0 + np.nan_to_num(net_np,   nan=0.0))

print("\n============================================================")
print("Long-Only Cross-Sectional Mean Reversion (Losers Only) + Hysteresis + MinHold + Costs")
print("------------------------------------------------------------")
print(f"ENTRY_Z: {ENTRY_Z} | EXIT_Z: {EXIT_Z} | MIN_HOLD_DAYS: {MIN_HOLD_DAYS}")
print(f"TC: {TC_BPS} bps per unit turnover")
print(f"Avg names held/day: {avg_names:.1f} (out of N={N})")
print(f"Avg daily turnover: {np.nanmean(turn_np):.3f}")
print("------------------------------------------------------------")
print("GROSS (no costs)")
print(f"APR: {stats_gross['apr']:+.3f} | Sharpe: {stats_gross['sharpe']:+.3f} | MaxDD: {stats_gross['max_dd']:+.3f}")
print("NET (after costs)")
print(f"APR: {stats_net['apr']:+.3f} | Sharpe: {stats_net['sharpe']:+.3f} | MaxDD: {stats_net['max_dd']:+.3f}")
print("------------------------------------------------------------")
print(f"Ending equity (gross/net): {eq_gross[-1]:.3f} / {eq_net[-1]:.3f}")
print("============================================================")

# Optional: latest holdings
last_day = dates[-1]
w_last = pd.Series(W_cpu[-1], index=px.columns)
print("\nLatest holdings (nonzero, held book):")
print(w_last[w_last.abs() > 0].sort_values(ascending=False).head(30))


Loaded universe: N=102, T=543, period 2024-01-02 -> 2026-03-03

Long-Only Cross-Sectional Mean Reversion (Losers Only) + Hysteresis + MinHold + Costs
------------------------------------------------------------
ENTRY_Z: 1.3 | EXIT_Z: 0.5 | MIN_HOLD_DAYS: 2
TC: 5 bps per unit turnover
Avg names held/day: 7.6 (out of N=102)
Avg daily turnover: 1.664
------------------------------------------------------------
GROSS (no costs)
APR: +0.676 | Sharpe: +1.683 | MaxDD: -0.282
NET (after costs)
APR: +0.466 | Sharpe: +1.161 | MaxDD: -0.306
------------------------------------------------------------
Ending equity (gross/net): 3.619 / 2.305

Latest holdings (nonzero, held book):
Ticker
STX     0.239217
MNST    0.169789
GEHC    0.166498
WDC     0.115910
MAR     0.110829
IDXX    0.102207
TEAM    0.057180
LRCX    0.038369
dtype: float32


# Factoring in Costs and Adding Train/Test

In [6]:
import numpy as np
import pandas as pd
import yfinance as yf
import requests
import warnings
import torch

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
ANNUALIZATION = 252

START_DATE = "2024-01-01"
CHUNK = 25
MIN_OBS_FRAC = 0.90

ENTRY_Z = 1.3
EXIT_Z  = 0.5
MIN_HOLD_DAYS = 2

TC_BPS = 5
TC = TC_BPS / 10000.0

EPS = 1e-6

TRAIN_FRAC = 0.70   # <<< NEW


tickers = [
    "NVDA","AAPL","MSFT","AMZN","TSLA","META","GOOGL","WMT","GOOG","AVGO",
    "MU","COST","NFLX","PLTR","AMD","CSCO","AMAT","LRCX","TMUS","LIN",
    "PEP","INTC","AMGN","KLAC","TXN","GILD","ISRG","ADI","HON","QCOM",
    "SHOP","BKNG","APP","ASML","VRTX","PANW","CEG","INTU","CMCSA","SBUX",
    "ADBE","CRWD","WDC","MELI","MAR","ADP","CDNS","REGN","CTAS","STX",
    "CSX","ORLY","MNST","SNPS","MDLZ","AEP","DASH","WBD","MRVL","PDD",
    "ROST","PCAR","BKR","FTNT","ABNB","NXPI","MPWR","FER",   # Ferrovial: US ADR is "FER"
    "FAST","ADSK","FANG","IDXX","EA","EXC","XEL","CCEP","AXON","TRI",
    "ODFL","ALNY","PYPL","KDP","MCHP","TTWO","ROP","CPRT","MSTR",  # Strategy Inc -> MicroStrategy ticker MSTR
    "GEHC","DDOG","PAYX","INSM","CTSH","CHTR","VRSK","KHC","DXCM","WDAY",
    "ZS","CSGP","ARM","CASH",  # CASH will fail on yfinance; it will be filtered out
    "TEAM"
]

def download_adj_close(tickers, start, chunk=25):
    """
    Downloads Adj Close for tickers in chunks to avoid yfinance throttling.
    Returns a DataFrame indexed by date, columns=tickers.
    """
    out = []
    for i in range(0, len(tickers), chunk):
        batch = tickers[i:i+chunk]
        df = yf.download(
            tickers=batch,
            start=start,
            auto_adjust=False,   # we explicitly take Adj Close
            progress=False,
            group_by="column",
            threads=True,
        )

        # yfinance returns:
        # - single ticker: columns like ['Open','High',...]
        # - multiple tickers: multiindex columns (field, ticker) OR (ticker, field) depending on settings
        if df.empty:
            continue

        if isinstance(df.columns, pd.MultiIndex):
            # try both common layouts
            if ("Adj Close" in df.columns.get_level_values(0)):
                # layout: (field, ticker)
                adj = df["Adj Close"].copy()
            elif ("Adj Close" in df.columns.get_level_values(1)):
                # layout: (ticker, field)
                adj = df.xs("Adj Close", axis=1, level=1).copy()
            else:
                raise ValueError("Couldn't find Adj Close in multiindex columns.")
        else:
            # single ticker case
            if "Adj Close" in df.columns:
                adj = df[["Adj Close"]].rename(columns={"Adj Close": batch[0]})
            else:
                raise ValueError("Couldn't find Adj Close in columns.")

        out.append(adj)

    if not out:
        return pd.DataFrame()

    px = pd.concat(out, axis=1)
    px = px.sort_index()
    px = px.loc[:, ~px.columns.duplicated()]
    return px
# =========================
# Prices -> Returns
# =========================
px = download_adj_close(tickers, start=START_DATE, chunk=CHUNK)

min_obs = int(MIN_OBS_FRAC * len(px))
keep = px.count()
keep = keep[keep >= min_obs].index
px = px[keep].ffill()

ret = px.pct_change()  # keep NaNs
dates = ret.index

ret_np = ret.to_numpy(dtype=np.float32)
mask_np = ~np.isnan(ret_np)
R_np = np.nan_to_num(ret_np, nan=0.0)

R = torch.tensor(R_np, dtype=torch.float32, device=device)     # [T, N]
M = torch.tensor(mask_np.astype(np.float32), device=device)    # [T, N]
T, N = R.shape

print(f"Loaded universe: N={N}, T={T}, period {dates.min().date()} -> {dates.max().date()}")

# =========================
# Cross-sectional z-scores per day (skipna)
# =========================
count = M.sum(dim=1, keepdim=True).clamp_min(1.0)
cs_mean = (R * M).sum(dim=1, keepdim=True) / count

dev = (R - cs_mean) * M
cs_var = (dev * dev).sum(dim=1, keepdim=True) / count
cs_std = torch.sqrt(cs_var).clamp_min(EPS)

Z = (dev / cs_std) * M  # [T, N]

# =========================
# Long-only hysteresis + min hold (losers only)
# =========================
W = torch.zeros((T, N), dtype=torch.float32, device=device)

active = torch.zeros((N,), dtype=torch.bool, device=device)
hold_days = torch.zeros((N,), dtype=torch.int16, device=device)

for t in range(T):
    z_t = Z[t]
    valid_t = (M[t] > 0)

    # increment hold counters for active
    hold_days = torch.where(active, hold_days + 1, torch.zeros_like(hold_days))

    # Entry: losers only
    enter = (~active) & valid_t & (z_t <= -ENTRY_Z)

    hold_days = torch.where(enter, torch.ones_like(hold_days), hold_days)
    active = active | enter

    # Exit eligibility after min hold
    can_exit = active & (hold_days >= MIN_HOLD_DAYS)

    # Exit: recovered above -EXIT_Z OR invalid
    exit_condition = (~valid_t) | (z_t > -EXIT_Z)
    exit_ = can_exit & exit_condition

    active = active & (~exit_)
    hold_days = torch.where(active, hold_days, torch.zeros_like(hold_days))

    # signal = -z on active (z negative => positive)
    signal = torch.zeros_like(z_t)
    signal[active] = -z_t[active]

    den = torch.sum(torch.abs(signal))
    if den > 0:
        W[t] = signal / den
    else:
        W[t] = 0.0

# =========================
# No lookahead
# =========================
W_lag = torch.zeros_like(W)
W_lag[1:] = W[:-1]

gross_ret = (W_lag * R).sum(dim=1)

# =========================
# Transaction costs: turnover on held book
# =========================
turnover = torch.abs(W_lag - torch.roll(W_lag, shifts=1, dims=0)).sum(dim=1)
turnover[0] = 0.0

tc_cost = TC * turnover
net_ret = gross_ret - tc_cost

# =========================
# Metrics
# =========================
def perf_stats(r_1d: np.ndarray):
    r = np.nan_to_num(r_1d, nan=0.0)
    r = r[np.isfinite(r)]
    if len(r) == 0:
        return dict(apr=np.nan, sharpe=np.nan, max_dd=np.nan, n=0)

    mu = r.mean()
    sd = r.std(ddof=1)
    sharpe = (mu / sd) * np.sqrt(ANNUALIZATION) if sd > 0 else np.nan
    apr = mu * ANNUALIZATION

    equity = np.cumprod(1.0 + r)
    running_max = np.maximum.accumulate(equity)
    max_dd = np.min(equity / running_max - 1.0)

    return dict(apr=apr, sharpe=sharpe, max_dd=max_dd, n=len(r))

# Convert to numpy (keep full length T)
gross_np = gross_ret.detach().cpu().numpy()
net_np   = net_ret.detach().cpu().numpy()
turn_np  = turnover.detach().cpu().numpy()

# =========================
# >>> NEW: Train/Test split by time index
# =========================
split = int(TRAIN_FRAC * T)
train_gross, test_gross = gross_np[:split], gross_np[split:]
train_net,   test_net   = net_np[:split],   net_np[split:]

train_end = dates[split-1].date() if split > 0 else None
test_start = dates[split].date() if split < len(dates) else None

# Stats buckets
stats = {
    "GROSS (no costs)": {
        "train": perf_stats(train_gross),
        "test":  perf_stats(test_gross),
        "all":   perf_stats(gross_np),
    },
    f"NET (after costs, TC={TC_BPS} bps/turnover)": {
        "train": perf_stats(train_net),
        "test":  perf_stats(test_net),
        "all":   perf_stats(net_np),
    }
}

# Diagnostics
W_cpu = W_lag.detach().cpu().numpy()
names_held = (np.abs(W_cpu) > 0).sum(axis=1)
avg_names = names_held.mean()

eq_gross = np.cumprod(1.0 + np.nan_to_num(gross_np, nan=0.0))
eq_net   = np.cumprod(1.0 + np.nan_to_num(net_np,   nan=0.0))

print("\n============================================================")
print("Long-Only Cross-Sectional MR (Losers) + Hysteresis + MinHold + Costs + Train/Test")
print("------------------------------------------------------------")
print(f"ENTRY_Z: {ENTRY_Z} | EXIT_Z: {EXIT_Z} | MIN_HOLD_DAYS: {MIN_HOLD_DAYS}")
print(f"TC: {TC_BPS} bps per unit turnover")
print(f"Avg names held/day: {avg_names:.1f} (out of N={N})")
print(f"Avg daily turnover: {np.nanmean(turn_np):.3f}")
print(f"Train end date: {train_end}")
print(f"Test start date: {test_start}")
print("------------------------------------------------------------")

for name, d in stats.items():
    print(name)
    for bucket in ["train", "test", "all"]:
        s = d[bucket]
        print(f"  {bucket.upper():5s} | APR {s['apr']:+.3f} | Sharpe {s['sharpe']:+.3f} | MaxDD {s['max_dd']:+.3f} | n={s['n']}")
    print("------------------------------------------------------------")

print(f"Ending equity (gross/net): {eq_gross[-1]:.3f} / {eq_net[-1]:.3f}")
print("============================================================")

# Optional: latest holdings
last_day = dates[-1]
w_last = pd.Series(W_cpu[-1], index=px.columns)
print("\nLatest holdings (nonzero, held book):")
print(w_last[w_last.abs() > 0].sort_values(ascending=False).head(30))


Loaded universe: N=102, T=543, period 2024-01-02 -> 2026-03-03

Long-Only Cross-Sectional MR (Losers) + Hysteresis + MinHold + Costs + Train/Test
------------------------------------------------------------
ENTRY_Z: 1.3 | EXIT_Z: 0.5 | MIN_HOLD_DAYS: 2
TC: 5 bps per unit turnover
Avg names held/day: 7.6 (out of N=102)
Avg daily turnover: 1.664
Train end date: 2025-07-09
Test start date: 2025-07-10
------------------------------------------------------------
GROSS (no costs)
  TRAIN | APR +0.760 | Sharpe +1.773 | MaxDD -0.282 | n=380
  TEST  | APR +0.477 | Sharpe +1.449 | MaxDD -0.166 | n=163
  ALL   | APR +0.675 | Sharpe +1.683 | MaxDD -0.282 | n=543
------------------------------------------------------------
NET (after costs, TC=5 bps/turnover)
  TRAIN | APR +0.550 | Sharpe +1.283 | MaxDD -0.306 | n=380
  TEST  | APR +0.269 | Sharpe +0.817 | MaxDD -0.192 | n=163
  ALL   | APR +0.466 | Sharpe +1.160 | MaxDD -0.306 | n=543
------------------------------------------------------------
En

# Trading Strategy Live on Alpaca

In [ ]:
import os
import json
import time
import math
import requests
import numpy as np
import pandas as pd
import pytz
import re
from io import StringIO
from datetime import datetime, timedelta

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

TZ = pytz.timezone("America/New_York")

STATE_PATH = "strategy_state.json"

# Strategy params (your exact logic)
ENTRY_Z = 1.3
EXIT_Z = 0.5
MIN_HOLD_DAYS = 2

API_KEY = ""
API_SECRET = ""

trading = TradingClient(API_KEY, API_SECRET, paper=True)

acct = trading.get_account()
print("Account status:", acct.status)
print("Buying power:", acct.buying_power)

# Portfolio / execution params
MIN_TRADE_DOLLARS = 50.0          # ignore tiny rebalances
MAX_POSITIONS = 200               # safety cap (optional)
ALLOW_FRACTIONAL = True           # Alpaca supports notional orders for fractionals
TIME_IN_FORCE = TimeInForce.DAY   # for market orders

# Data params
LOOKBACK_DAYS = 30                # only need enough to compute yesterday return; kept modest for reliability
BAR_LIMIT_PADDING = 5             # extra buffer for weekends/holidays

def _clean_ticker_list(raw):
    """Normalize tickers for yfinance/alpaca style symbols."""
    out = []
    for t in raw:
        if t is None:
            continue
        s = str(t).strip().upper()
        if not s or s in {"NAN", "NONE"}:
            continue

        # Common symbol normalization:
        s = s.replace(".", "-")             # BRK.B -> BRK-B
        s = s.replace(" ", "")              # remove spaces
        s = s.replace("/", "-")             # some feeds use BRK/B
        s = s.replace("_", "-")

        # remove obvious non-ticker junk
        if any(x in s for x in ["CASH", "USD", "US DOLLAR", "CURRENCY", "EQUIVALENT"]):
            continue

        # allow letters, numbers, dash (e.g., BRK-B)
        if re.fullmatch(r"[A-Z0-9\-]{1,10}", s):
            out.append(s)

    return sorted(set(out))


def get_qqq_holdings_tickers():
    """
    Attempts to fetch current QQQ holdings tickers.
    1) Tries an Invesco holdings endpoint (if available).
    2) Falls back to Nasdaq-100 constituents from Wikipedia.
    """

    # ---- Try Invesco QQQ holdings endpoints (these can change) ----
    urls_to_try = [
        # Invesco sometimes provides a holdings CSV via the product/holdings page.
        # These URLs may change; if they fail, fallback still works.
        "https://www.invesco.com/us/financial-products/etfs/holdings?audienceType=Investor&ticker=QQQ",
        "https://www.invesco.com/us/financial-products/etfs/holdings/main/holdings?audienceType=Investor&ticker=QQQ",
    ]

    headers = {"User-Agent": "Mozilla/5.0"}

    for url in urls_to_try:
        try:
            r = requests.get(url, headers=headers, timeout=20)
            r.raise_for_status()

            # Look for a CSV link in the HTML (best effort)
            html = r.text
            # Common patterns: ".csv", "download", "holdings", etc.
            csv_links = re.findall(r'https?://[^"\']+\.csv[^"\']*', html, flags=re.IGNORECASE)
            csv_links = [x for x in csv_links if "hold" in x.lower() or "constitu" in x.lower()]

            for csv_url in csv_links[:5]:
                try:
                    rcsv = requests.get(csv_url, headers=headers, timeout=20)
                    rcsv.raise_for_status()
                    df = pd.read_csv(StringIO(rcsv.text))

                    # Try common ticker columns
                    possible_cols = ["Ticker", "Holding Ticker", "Symbol", "Security Ticker", "Identifier"]
                    col = next((c for c in possible_cols if c in df.columns), None)
                    if col is None:
                        continue

                    tickers = _clean_ticker_list(df[col].tolist())

                    # Nasdaq-100 should be ~100 names (give a range to be safe)
                    if 80 <= len(tickers) <= 120:
                        return tickers
                except Exception:
                    continue
        except Exception:
            pass

    # ---- Fallback: Nasdaq-100 constituents from Wikipedia (proxy for QQQ) ----
    wiki_url = "https://en.wikipedia.org/wiki/Nasdaq-100"
    try:
        html = requests.get(wiki_url, headers=headers, timeout=20).text
        tables = pd.read_html(html)

        # Find the table that contains tickers (column often named "Ticker" or "Company" + "Ticker")
        best = None
        for t in tables:
            cols = [c.lower() for c in t.columns.astype(str)]
            if any("ticker" in c for c in cols) or any("symbol" in c for c in cols):
                best = t
                break

        if best is None:
            raise ValueError("Could not locate a ticker table on Nasdaq-100 Wikipedia page.")

        # Identify ticker column
        ticker_col = None
        for c in best.columns:
            cl = str(c).lower()
            if "ticker" in cl or "symbol" in cl:
                ticker_col = c
                break

        if ticker_col is None:
            # sometimes the first column is tickers
            ticker_col = best.columns[0]

        tickers = _clean_ticker_list(best[ticker_col].tolist())
        if len(tickers) < 70:
            raise ValueError("Fallback ticker list too small; Wikipedia format may have changed.")
        return tickers

    except Exception as e:
        raise RuntimeError(f"Failed to fetch QQQ/Nasdaq-100 tickers. Last error: {e}")

def download_adj_close(tickers, start, chunk=25):
    """
    Downloads Adj Close for tickers in chunks to avoid yfinance throttling.
    Returns a DataFrame indexed by date, columns=tickers.
    """
    out = []
    for i in range(0, len(tickers), chunk):
        batch = tickers[i:i+chunk]
        df = yf.download(
            tickers=batch,
            start=start,
            auto_adjust=False,   # we explicitly take Adj Close
            progress=False,
            group_by="column",
            threads=True,
        )

        # yfinance returns:
        # - single ticker: columns like ['Open','High',...]
        # - multiple tickers: multiindex columns (field, ticker) OR (ticker, field) depending on settings
        if df.empty:
            continue

        if isinstance(df.columns, pd.MultiIndex):
            # try both common layouts
            if ("Adj Close" in df.columns.get_level_values(0)):
                # layout: (field, ticker)
                adj = df["Adj Close"].copy()
            elif ("Adj Close" in df.columns.get_level_values(1)):
                # layout: (ticker, field)
                adj = df.xs("Adj Close", axis=1, level=1).copy()
            else:
                raise ValueError("Couldn't find Adj Close in multiindex columns.")
        else:
            # single ticker case
            if "Adj Close" in df.columns:
                adj = df[["Adj Close"]].rename(columns={"Adj Close": batch[0]})
            else:
                raise ValueError("Couldn't find Adj Close in columns.")

        out.append(adj)

    if not out:
        return pd.DataFrame()

    px = pd.concat(out, axis=1)
    px = px.sort_index()
    px = px.loc[:, ~px.columns.duplicated()]
    return px

# =========================
# HELPERS: STATE
# =========================
def load_state():
    if not os.path.exists(STATE_PATH):
        return {"asof_date": None, "positions": {}}  # positions: {SYM: {"active": bool, "hold_days": int}}
    with open(STATE_PATH, "r") as f:
        return json.load(f)

def save_state(state):
    with open(STATE_PATH, "w") as f:
        json.dump(state, f, indent=2, sort_keys=True)

# =========================
# HELPERS: ALPACA DATA
# =========================
def fetch_daily_closes(data_client, symbols, start_dt, end_dt):
    req = StockBarsRequest(
        symbol_or_symbols=symbols,
        timeframe=TimeFrame.Day,
        start=start_dt,
        end=end_dt,
        adjustment="all",
        feed="iex",   # <-- IMPORTANT: avoids SIP restriction
    )
    bars = data_client.get_stock_bars(req).df
    if bars is None or len(bars) == 0:
        return pd.DataFrame()

    close = bars["close"].unstack(level=0).sort_index()
    close.index = close.index.tz_convert(TZ)
    return close

# =========================
# STRATEGY: compute z for yesterday
# =========================
def compute_yesterday_zscores(close_df):
    """
    Using daily closes, compute yesterday's return per symbol,
    then cross-sectional mean/std and z-scores for that day.
    Returns (asof_date, z_series) where z_series is indexed by symbol.
    """
    if close_df.shape[0] < 2:
        return None, None

    # Use last two available trading days in the data
    last_date = close_df.index[-1]
    prev_date = close_df.index[-2]

    # daily returns for "yesterday" = close(last)/close(prev)-1
    r = close_df.iloc[-1] / close_df.iloc[-2] - 1.0
    r = r.replace([np.inf, -np.inf], np.nan).dropna()

    if len(r) < 50:
        return None, None

    mu = r.mean()
    sd = r.std(ddof=1)
    if not np.isfinite(sd) or sd <= 0:
        return None, None

    z = (r - mu) / sd
    z = z.replace([np.inf, -np.inf], np.nan).dropna()

    # asof_date is the date the signal is based on (yesterday's close day)
    asof_date = last_date.date().isoformat()
    return asof_date, z

# =========================
# STRATEGY: update active set with hysteresis + min hold
# Long-only losers:
#   enter if z <= -ENTRY_Z
#   exit (only if hold_days >= MIN_HOLD_DAYS) if z > -EXIT_Z
# =========================
def update_positions_state(state, universe, asof_date, z):
    """
    Updates state['positions'] based on today's (asof_date) z-scores.
    Returns list of active symbols and their signals (-z).
    """
    pos = state.get("positions", {})
    prev_asof = state.get("asof_date")

    # Increment hold days once per new asof_date
    is_new_day = (prev_asof != asof_date)

    # Ensure universe present in state (optional)
    for sym in universe:
        if sym not in pos:
            pos[sym] = {"active": False, "hold_days": 0}

    # If new day, increment hold days for active positions
    if is_new_day:
        for sym, st in pos.items():
            if st["active"]:
                st["hold_days"] += 1

    # Now apply entry/exit rules using current z scores
    z_dict = z.to_dict()

    for sym in universe:
        st = pos[sym]
        zi = z_dict.get(sym, None)

        # If we don't have z today for a symbol, treat as invalid => exit if eligible
        valid = (zi is not None and np.isfinite(zi))

        if not st["active"]:
            # ENTRY: losers only
            if valid and zi <= -ENTRY_Z:
                st["active"] = True
                st["hold_days"] = 1  # start counting from entry day
        else:
            # EXIT: only after min hold
            if st["hold_days"] >= MIN_HOLD_DAYS:
                if (not valid) or (zi > -EXIT_Z):
                    st["active"] = False
                    st["hold_days"] = 0

    # Build active signals: signal = -z (positive for losers since z is negative)
    active_syms = []
    signals = []
    for sym in universe:
        st = pos[sym]
        zi = z_dict.get(sym, None)
        if st["active"] and zi is not None and np.isfinite(zi):
            active_syms.append(sym)
            signals.append(-float(zi))  # -z, positive number

    state["positions"] = pos
    state["asof_date"] = asof_date

    return active_syms, np.array(signals, dtype=np.float64)

def signals_to_target_weights(active_syms, signals):
    """
    Long-only weights proportional to signal, sum to 1.
    """
    if len(active_syms) == 0:
        return {}

    signals = np.where(np.isfinite(signals), signals, 0.0)
    signals = np.clip(signals, 0.0, None)

    s = signals.sum()
    if s <= 0:
        return {}

    w = signals / s
    target = {sym: float(wi) for sym, wi in zip(active_syms, w)}
    return target

# =========================
# EXECUTION: rebalance paper account to target weights
# =========================
def rebalance_notional_fixed(
    trading_client,
    target_w: dict,
    desired_hold: set,
    min_trade_dollars: float = 50.0,
    allow_resize_held: bool = True,
    state_positions: dict | None = None,
    min_hold_days: int = 2,
):
    """
    FIXED rebalance logic that respects your state/min-hold:

    Inputs:
      - target_w: dict {symbol: weight} for symbols with valid z today.
                 weights sum to ~1 across the active tradable set that had valid z.
      - desired_hold: set of symbols that MUST remain held according to strategy state
                      (active=True), even if z is missing today.
      - allow_resize_held:
          True  -> allow resizing held names toward target (may trade small deltas)
          False -> if a symbol is in desired_hold and hold_days < min_hold_days, do NOT resize it
                   (prevents ANY sells/buys during the min hold window)
      - state_positions: your state["positions"] dict (needed only if allow_resize_held=False)
      - min_hold_days: your MIN_HOLD_DAYS setting (default 2)

    What it does:
      1) Close positions that strategy says should NOT be held (not in desired_hold).
      2) For symbols in target_w, adjust toward target dollars.
    """

    # 0) Pull account equity (used to convert weights -> target dollars)
    acct = trading_client.get_account()
    equity = float(acct.equity)

    # 1) Pull current positions (market value $)
    positions = trading_client.get_all_positions()
    current_mv = {p.symbol: float(p.market_value) for p in positions}

    # ----------------------------
    # 2) CLOSE ONLY if strategy says NOT to hold
    # ----------------------------
    for sym, mv in current_mv.items():
        if sym not in desired_hold and abs(mv) >= min_trade_dollars:
            # This is a FULL close. Allowed because strategy state says inactive.
            trading_client.close_position(sym)

    # ----------------------------
    # 3) Resize toward target weights (only for symbols with target_w)
    # ----------------------------
    for sym, w in target_w.items():
        # Convert weight -> target notional $
        tgt = float(w) * equity

        # Current market value ($) (0 if not held yet)
        cur = current_mv.get(sym, 0.0)

        # Desired change in dollars
        delta = tgt - cur

        # Round to 2 decimals to satisfy Alpaca (notional must be 2dp)
        notional = round(abs(delta), 2)

        if notional < min_trade_dollars:
            continue

        # Optional: freeze resizing for names still in min-hold window
        if not allow_resize_held and sym in desired_hold and state_positions is not None:
            st = state_positions.get(sym, {"active": False, "hold_days": 0})
            if st.get("active", False) and st.get("hold_days", 0) < min_hold_days:
                # Skip ANY buy/sell resizing during the min hold window
                continue

        side = OrderSide.BUY if delta > 0 else OrderSide.SELL

        order = MarketOrderRequest(
            symbol=sym,
            notional=notional,
            side=side,
            time_in_force=TIME_IN_FORCE
        )
        trading_client.submit_order(order_data=order)

# =========================
# LIVE PNL MONITOR (optional)
# =========================
def print_live_pnl(trading_client, interval_sec=10):
    while True:
        acct = trading_client.get_account()
        equity = float(acct.equity)
        cash = float(acct.cash)
        pv = float(acct.portfolio_value)

        print("\n--- PAPER ACCOUNT ---")
        print("Equity:", equity, "Cash:", cash, "Portfolio:", pv)

        pos = trading_client.get_all_positions()
        if not pos:
            print("(no positions)")
        else:
            print("--- POSITIONS ---")
            for p in pos:
                print(f"{p.symbol:6s} qty={p.qty:>8s} mv={p.market_value:>10s} "
                      f"unreal={p.unrealized_pl:>10s} unreal%={p.unrealized_plpc:>10s}")

        time.sleep(interval_sec)

def filter_to_alpaca_tradable(trading_client, symbols):
    """
    Keep only symbols that Alpaca says are active + tradable.
    This avoids 'invalid symbol' errors in data requests.
    """
    assets = trading_client.get_all_assets()
    tradable = set()
    for a in assets:
        # For stocks, you want active + tradable
        if a.status == "active" and a.tradable:
            tradable.add(a.symbol)

    symbols = [s for s in symbols if s in tradable]
    return symbols

data = StockHistoricalDataClient(API_KEY, API_SECRET)

# 1) Universe = SPY holdings (best-effort)
tickers = get_qqq_holdings_tickers()
universe = tickers
print(f"Universe size (raw): {len(universe)}")

universe = filter_to_alpaca_tradable(trading, universe)
print("Universe tradable on Alpaca:", len(universe))

# Optional: safety cap
universe = universe[:1000]

# 2) Pull daily closes from Alpaca for universe
# We'll request a short lookback window; last two trading days are enough for yesterday return.
end_dt = datetime.now(TZ)
start_dt = end_dt - timedelta(days=LOOKBACK_DAYS + BAR_LIMIT_PADDING)

# Alpaca data endpoints handle many symbols, but large universes may need chunking.
# We'll chunk for reliability.
closes_parts = []
chunk = 200
for i in range(0, len(universe), chunk):
    batch = universe[i:i+chunk]
    close = fetch_daily_closes(data, batch, start_dt, end_dt)
    if close is not None and len(close) > 0:
        closes_parts.append(close)

if not closes_parts:
    raise RuntimeError("No close data returned from Alpaca. Check market data permissions/subscription.")

close_df = pd.concat(closes_parts, axis=1)
close_df = close_df.loc[:, ~close_df.columns.duplicated()]
close_df = close_df.sort_index()

# 3) Compute yesterday z-scores from last two trading days available
asof_date, z = compute_yesterday_zscores(close_df)
if asof_date is None:
    raise RuntimeError("Not enough data to compute z-scores (need at least 2 trading days and enough symbols).")

print(f"Signal as-of close date: {asof_date} | z-count: {len(z)}")

# 4) Load state, update active set, build target weights
state = load_state()
active_syms, signals = update_positions_state(state, universe=close_df.columns.tolist(), asof_date=asof_date, z=z)

# Optional safety: cap max positions to keep execution manageable
if len(active_syms) > MAX_POSITIONS:
    # pick strongest signals (largest -z)
    idx = np.argsort(-signals)[:MAX_POSITIONS]
    active_syms = [active_syms[j] for j in idx]
    signals = signals[idx]

target_w = signals_to_target_weights(active_syms, signals)
save_state(state)

print(f"Active positions today: {len(target_w)}")
if len(target_w) > 0:
    top = sorted(target_w.items(), key=lambda x: -x[1])[:15]
    print("Top target weights:", top)

# 5) Execute paper rebalance
today_universe = set(close_df.columns.tolist())
desired_hold = {sym for sym, st in state["positions"].items()
                if st.get("active") and sym in today_universe}

rebalance_notional_fixed(
    trading_client=trading,
    target_w=target_w,
    desired_hold=desired_hold,
    min_trade_dollars=MIN_TRADE_DOLLARS,
    allow_resize_held=True,             # default behavior
    state_positions=state["positions"], # only needed if allow_resize_held=False
    min_hold_days=MIN_HOLD_DAYS
)

print("Rebalance orders submitted (paper).")

# 6) Optional live PnL monitor
monitor = False
if monitor:
    print_live_pnl(trading, interval_sec=10)

Account status: AccountStatus.ACTIVE
Buying power: 200000
Universe size (raw): 101
Universe tradable on Alpaca: 101
Signal as-of close date: 2026-03-04 | z-count: 101
Active positions today: 5
Top target weights: [('DXCM', 0.3253744833724153), ('FANG', 0.3139048762917318), ('TMUS', 0.12144801077038744), ('GOOG', 0.12143024370941485), ('GOOGL', 0.11784238585605059)]
Rebalance orders submitted (paper).
